# Sparse Primitive Flow Composition Demo

This demo runs the simplified PrimitiveFlow method: normal denoising steps use the full target prompt `P`; selected timesteps aggregate source `S0`, primitive prompts `S1...Sk`, and `P`; LTP uses `P` as the reference. No VQA, reward model, external judge, training, or fine-tuning is used.

In [1]:
GITHUB_REPO_URL = "https://github.com/Soobiwan/aim-flow.git"
HF_TOKEN = ""
PROMPT_SECTION = "primitive_flow_prompts"
PROMPT_KEY = "german_shepherd"
OUTPUT_DIR = "/kaggle/working/primitive_flow_outputs/german_shepherd"

In [2]:
!git clone {GITHUB_REPO_URL} /kaggle/working/aim-flow
%cd /kaggle/working/aim-flow

Cloning into '/kaggle/working/aim-flow'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 127 (delta 61), reused 97 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 71.15 KiB | 5.47 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/kaggle/working/aim-flow


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install --no-cache-dir --force-reinstall torch==2.4.1+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -r requirements-kaggle.txt

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:


In [4]:
from kaggle_secrets import UserSecretsClient
import os

if not HF_TOKEN:
    HF_TOKEN = UserSecretsClient().get_secret("Huggingface")
os.environ["HF_TOKEN"] = HF_TOKEN

In [5]:
import torch, diffusers, transformers
print("torch", torch.__version__)
print("diffusers", diffusers.__version__)
print("transformers", transformers.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU", props.name)
    print("VRAM GB", round(props.total_memory / 1024**3, 2))
else:
    print("CUDA not available")

torch 2.10.0+cu128
diffusers 0.36.0
transformers 5.0.0
GPU Tesla P100-PCIE-16GB
VRAM GB 15.89


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


In [6]:
!python scripts/run_compare.py \
  --config configs/sd3_medium_kaggle.yaml \
  --prompts configs/sample_prompts.yaml \
  --prompt-section {PROMPT_SECTION} \
  --prompt-key {PROMPT_KEY} \
  --output-dir {OUTPUT_DIR}_final_only \
  --modes base source_only primitive_flow_final_only \
  --final-only

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()
Traceback (most recent call last):
  Fil

In [7]:
!python scripts/run_compare.py \
  --config configs/sd3_medium_kaggle.yaml \
  --prompts configs/sample_prompts.yaml \
  --prompt-section {PROMPT_SECTION} \
  --prompt-key {PROMPT_KEY} \
  --output-dir {OUTPUT_DIR}_sparse \
  --modes base source_only primitive_flow_sparse \
  --aggregation-steps 12 16 20 23

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()
Traceback (most recent call last):
  Fil

In [8]:
from IPython.display import Image, display
display(Image(filename=f"{OUTPUT_DIR}_sparse/comparison_grid.png"))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/primitive_flow_outputs/german_shepherd_sparse/comparison_grid.png'

In [ ]:
import json
metadata_path = f"{OUTPUT_DIR}_sparse/metadata_primitive_flow_sparse.json"
with open(metadata_path) as f:
    meta = json.load(f)
print("target:", meta["target_prompt"])
print("source:", meta["source_prompt"])
print("primitives:", [p["text"] for p in meta["primitive_prompts"]])
print("aggregation_steps:", meta["aggregation_steps"])
print("ltp_mode:", meta["ltp_mode"])
print("fallback:", any(step.get("ltp_fallback") for step in meta["debug_steps"] if step.get("do_aggregate")))
for step in meta["debug_steps"]:
    if step.get("do_aggregate"):
        print(step["step_index"], step.get("softmax_weights"))

## Troubleshooting

- Reduce size: `--height 384 --width 384`
- Reduce steps: `--num-inference-steps 16`
- Use velocity LTP: `--ltp-mode velocity`
- Reduce primitives: `--max-primitives 2`
- Try final-only first: `--final-only`
- For 16 steps, try aggregation steps `8 12 15`